# 💬 멀티턴 대화 구현 + 품질 불일치 관찰

---

## 🎯 오늘의 상황

> 파라미터 실험까지 마친 법률 AI 스타트업 개발자.  
> 팀장이 말합니다: "EXAONE이 이전 대화를 기억하질 못해요.  
> '아까 말한 내용 요약해줘' 하면 '무슨 내용인지 모르겠다'고 합니다."
>
> 문제의 원인은 **싱글턴 구조**입니다.  
> 오늘은 이전 대화를 기억하는 **멀티턴 챗봇**을 직접 만들어봅니다.

---

## 📋 실습 순서

```
STEP 1. 싱글턴 vs 멀티턴 차이 확인
STEP 2. chat() 함수 — 멀티턴의 핵심
STEP 3. safe_generate — 실패해도 안전하게
STEP 4. 품질 불일치 관찰 실험
STEP 5. 히스토리 관리 주의사항
```

---

## 💡 핵심 개념

> **LLM은 메모리가 없다!**  
> EXAONE이 맥락을 기억하는 것처럼 보이는 건  
> 이전 대화를 **매번 통째로 다시 보내기** 때문입니다.

| 구조 | messages 구성 | 맥락 기억 | 토큰 비용 |
|------|--------------|----------|----------|
| **싱글턴** | 매번 새로 생성 | ❌ 없음 | 일정 |
| **멀티턴** | 이전 내용 누적 | ✅ 있음 | 턴마다 증가 |

---
## 🔧 사전 준비

In [ ]:
# 패키지 설치 (처음 실행 시만)
!pip install -q --upgrade transformers accelerate

In [ ]:
import os
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
model_name = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"

print("📥 토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

print("📥 모델 로드 중... (약 2~3분 소요)")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
)
print(f"✅ 모델 로드 완료! GPU 메모리: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
# generate_response 함수 + 토큰 추적기 (이전 실습에서 사용한 것)
def generate_response(messages, max_new_tokens=300, temperature=0.3, top_p=0.9):
    """EXAONE 로컬 추론 — 메시지 리스트를 받아 응답 dict를 반환한다."""
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    input_ids = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).input_ids.to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True if temperature > 0 else False,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][input_ids.shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return {
        "content": answer,
        "input_tokens": input_ids.shape[1],
        "output_tokens": len(generated_ids),
    }


class TokenTracker:
    def __init__(self):
        self.total_input  = 0
        self.total_output = 0
        self.call_count   = 0

    def add(self, result):
        self.total_input  += result["input_tokens"]
        self.total_output += result["output_tokens"]
        self.call_count   += 1

    def report(self):
        api_cost = (self.total_input * 0.15 + self.total_output * 0.60) / 1_000_000
        print(f"총 {self.call_count}회 호출 | 입력 {self.total_input:,} + 출력 {self.total_output:,} 토큰")
        print(f"로컬 추론 비용: ₩0")
        print(f"(참고) API 환산 비용: ${api_cost:.4f} (약 {api_cost * 1400:.1f}원)")


tracker = TokenTracker()
print("✅ 준비 완료!")

---
## STEP 1. 싱글턴 vs 멀티턴 차이 확인

먼저 **싱글턴이 왜 문제인지** 직접 확인합니다.  
"방금 말한 내용 요약해줘" 라고 했을 때 어떻게 다른지 보세요.

In [ ]:
# ─── 싱글턴 방식 ─── 매번 새로운 messages
system = {"role": "system", "content": "당신은 법률 전문 AI입니다."}

# 첫 번째 질문
r1 = generate_response(
    messages=[system, {"role": "user", "content": "계약서에서 가장 중요한 조항은?"}],
    temperature=0.3, max_new_tokens=200,
)
print("[싱글턴 1턴] 계약서 조항 질문")
print(r1["content"][:150])

# 두 번째 질문 — 맥락을 이어가려 했지만...
r2 = generate_response(
    messages=[system, {"role": "user", "content": "방금 말한 내용을 요약해 주세요."}],  # 새 messages
    temperature=0.3, max_new_tokens=200,
)
print("\n[싱글턴 2턴] 요약 요청 → 이전 대화를 모릅니다!")
print(r2["content"][:150])

tracker.add(r1)
tracker.add(r2)

---
## STEP 2. chat() 함수 — 멀티턴의 핵심

멀티턴의 비결은 단순합니다.  
**대화할 때마다 이전 내용을 리스트에 쌓아서 함께 보내는 것**입니다.

```
초기:  [system]
1턴:   [system, user_1]                       → 추론 → assistant_1 추가
2턴:   [system, user_1, assistant_1, user_2]  → 추론 → assistant_2 추가
```

In [ ]:
# ① 대화 히스토리 초기화 — system 프롬프트로 시작
conversation_history = [
    {"role": "system", "content": "당신은 법률 전문 AI입니다. 쉽게 설명하세요."}
]

# ② chat 함수 — 4단계로 동작
def chat(user_input):
    # 1단계: user 메시지를 히스토리에 추가
    conversation_history.append({"role": "user", "content": user_input})

    # 2단계: 히스토리 전체를 모델에 전달 (이게 핵심!)
    result = generate_response(
        messages=conversation_history,
        temperature=0.3,
        max_new_tokens=300,
    )

    # 3단계: 응답을 히스토리에 추가
    reply = result["content"]
    conversation_history.append({"role": "assistant", "content": reply})

    # 4단계: 토큰 추적 + 응답 반환
    tracker.add(result)
    return reply

print("✅ chat() 함수 준비 완료")

In [ ]:
# ③ 3턴 대화 실행 — 세 번째 질문에서 맥락이 연결되는지 확인!
print("=" * 50)
print("[1턴]", chat("계약서에서 가장 중요한 조항이 무엇인가요?"))

print("\n" + "=" * 50)
print("[2턴]", chat("그렇다면 해지 조항은 어떻게 작성해야 하나요?"))

print("\n" + "=" * 50)
print("[3턴]", chat("방금 말한 내용을 요약해 주세요."))  # ← 1·2턴 내용을 참조해야 함!

In [ ]:
# ④ 히스토리 구조 확인 — 어떻게 쌓였는지 눈으로 보기
print(f"📊 히스토리 길이: {len(conversation_history)}개")
print(f"   → system(1) + user·assistant × 3턴 = 7개")

# role 순서 출력
roles = [item["role"] for item in conversation_history]
print(f"\n📋 role 순서: {roles}")

# 입력 토큰이 턴마다 얼마나 늘어나는지 체감
print("\n💡 멀티턴 입력 토큰 누적 구조:")
print("  1턴: system + user_1 만 전달")
print("  2턴: system + user_1 + assistant_1 + user_2 전달 (이전 내용 포함!)")
print("  3턴: system + ... + user_3 전달 (더 길어짐)")

---
## STEP 3. safe_generate — 실패해도 안전하게

로컬 추론은 언제든 실패할 수 있습니다. (GPU 메모리 부족, 토큰 초과 등)  
실패했을 때 **히스토리가 망가지지 않도록** 안전 장치를 만들어봅니다.

> 💡 **API vs 로컬 재시도 전략의 차이**  
> - API: 네트워크·Rate Limit 오류 → 지수 백오프 (1→2→4초 대기)  
> - 로컬: GPU 메모리 오류가 주원인 → **GPU 캐시 정리 후 즉시 재시도**

In [ ]:
# ① 안전한 추론 함수 — GPU 오류 시 캐시 정리 후 재시도
def safe_generate(messages, temperature=0.3, max_new_tokens=300, max_retries=2):
    """로컬 추론 실패 시 GPU 캐시 정리 후 자동 재시도, 그래도 안 되면 None 반환"""

    for attempt in range(max_retries):
        try:
            result = generate_response(
                messages=messages,
                temperature=temperature,
                max_new_tokens=max_new_tokens,
            )
            tracker.add(result)          # 성공 시 토큰 추적
            return result["content"]     # 성공 시 응답 반환

        except Exception as e:
            print(f"⚠️ [{attempt+1}/{max_retries}] 오류: {e}")
            if attempt < max_retries - 1:
                print("   → GPU 메모리 정리 후 재시도...")
                torch.cuda.empty_cache()

    print("❌ 최대 재시도 횟수 초과")
    return None   # 모두 실패하면 None 반환

print("✅ safe_generate() 준비 완료")

In [ ]:
# ② safe 버전 chat 함수 — 실패 시 히스토리 복구
def chat_safe(user_input):
    # user 메시지를 히스토리에 추가
    conversation_history.append({"role": "user", "content": user_input})

    # 안전하게 추론 호출
    reply = safe_generate(conversation_history)

    if reply is None:
        conversation_history.pop()   # 실패 시 방금 추가한 user 메시지 제거 (히스토리 보호)
        return "⚠️ 응답을 받지 못했습니다. 다시 시도해 주세요."

    conversation_history.append({"role": "assistant", "content": reply})
    return reply

print("✅ chat_safe() 준비 완료")

---
## STEP 4. 품질 불일치 관찰 실험

**같은 질문, 같은 설정으로 3번 호출하면 항상 같은 품질의 답이 나올까요?**

답은 **아닙니다**. 이 현상을 직접 관찰하고 기록합니다.  
이 기록은 **Day 2 프롬프트 기법 실험의 입력 데이터**로 사용됩니다.

> ⚠️ **이 실험 결과는 반드시 저장하세요! Day 2에서 사용합니다.**

In [ ]:
# 같은 질문을 temperature=0.3으로 3번 호출
test_question = "임차인이 보증금을 돌려받지 못할 때 법적으로 어떤 조치를 취할 수 있나요?"

quality_samples = []   # 3개 응답을 저장할 리스트

for i in range(3):
    result = generate_response(
        messages=[
            {"role": "system", "content": "당신은 법률 전문 AI입니다."},
            {"role": "user",   "content": test_question}
        ],
        temperature=0.3,     # 낮은 값이지만 완전히 같지 않을 수 있음
        max_new_tokens=300,
    )
    answer = result["content"]
    quality_samples.append(answer)
    tracker.add(result)

    print(f"{'─'*50}")
    print(f"[{i+1}회차] ({result['output_tokens']}토큰)")
    print(answer[:200])

In [ ]:
# 3개 응답의 차이를 자동으로 분석
print("📋 품질 불일치 분석")
print("=" * 50)

for i, ans in enumerate(quality_samples):
    has_law       = "조" in ans or "법" in ans                          # 법조문 인용 여부
    has_structure = any(c in ans for c in ["1.", "①", "첫째", "- "])   # 구조화 여부

    print(f"\n[{i+1}회차] 길이: {len(ans)}자")
    print(f"  법조문 인용: {'✅' if has_law else '❌'}  구조화: {'✅' if has_structure else '❌'}")
    print(f"  시작: {ans[:60]}...")

# 동일 여부 확인
if quality_samples[0] == quality_samples[1] == quality_samples[2]:
    print("\n→ 3회 모두 동일")
else:
    print("\n→ ⚠️ 답변이 다릅니다! 이것이 '품질 불일치' 현상입니다.")

In [ ]:
# 관찰 결과를 JSON으로 저장 → Day 2에서 사용
# Drive 마운트 시도 (실패하면 로컬에 저장)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    save_dir = "/content/drive/MyDrive/llm_course/day1"
except Exception as e:
    print(f"⚠️ Drive 마운트 실패: {e}")
    save_dir = "/content"

os.makedirs(save_dir, exist_ok=True)

quality_note = {
    "question": test_question,
    "model": "EXAONE-3.5-2.4B-Instruct",
    "temperature": 0.3,
    "samples": quality_samples,
    "observation": "여기에 관찰 내용을 직접 작성하세요"   # ← 직접 수정!
}

with open(f"{save_dir}/quality_note.json", "w", encoding="utf-8") as f:
    json.dump(quality_note, f, ensure_ascii=False, indent=2)

print(f"✅ 저장 완료: {save_dir}/quality_note.json")
print("📌 Day 2에서 이 파일을 불러와 프롬프트 기법 실험을 시작합니다.")

**📝 품질 불일치 기록표 — 직접 채우세요 (Day 2 입력 데이터)**

| 항목 | 1회차 | 2회차 | 3회차 |
|------|-------|-------|-------|
| 법조문 인용 여부 | | | |
| 답변 구조 (목록형/서술형) | | | |
| 답변 길이 (글자 수) | | | |
| 정확도 (상/중/하) | | | |
| 핵심 차이점 | | | |

---
## STEP 5. 히스토리 관리 주의사항

멀티턴을 쓸 때 주의할 두 가지 문제입니다.

### ① 토큰 누적 — 턴이 늘수록 입력 토큰이 쌓입니다

| 턴 수 | 입력 토큰 (예시) |
|-------|----------------|
| 1턴 | ~100 토큰 |
| 3턴 | ~400 토큰 (4배) |
| 10턴 | ~1,500 토큰 (15배) |

> 💡 EXAONE은 로컬 추론이므로 토큰 비용은 없지만,  
> 입력이 길어질수록 **추론 시간**이 늘어납니다.

### ② 컨텍스트 윈도우 한계

| 모델 | 최대 길이 | 한국어 기준 |
|------|----------|------------|
| gpt-4o-mini | 128,000 토큰 | 약 4만 글자 |
| **EXAONE 3.5** | **32,768 토큰** | **약 1만 글자** |

> ⚠️ 긴 대화를 이어갈 때는 컨텍스트 윈도우를 초과하지 않도록 주의하세요!

In [ ]:
# 새 대화를 시작할 때 히스토리 초기화
def reset_conversation():
    global conversation_history
    conversation_history = [
        {"role": "system", "content": "당신은 법률 전문 AI입니다. 쉽게 설명하세요."}
    ]
    print("🔄 대화가 초기화되었습니다.")

reset_conversation()   # 초기화 확인

In [ ]:
# 전체 실습 누적 토큰 확인
print("💰 오늘 실습 전체 토큰 사용량")
tracker.report()

---
## 🔜 다음 시간 예고

> **지금까지 만든 것:** 멀티턴 구조 + 안전 추론 + 품질 불일치 관찰  
> **다음 시간:** 이것들을 합쳐서 **법률 챗봇 v0.1 완성**  
> **Day 2:** 오늘 저장한 품질 불일치 사례로 **프롬프트 기법 실험**
>
> "프롬프트만 바꾸면 품질 불일치가 해결될까?  
> 안 된다면? → Day 4 파인튜닝이 그 답입니다."

> 💡 EXAONE은 로컬 추론이므로 API 비용 없이 실험할 수 있습니다.  
> Day 4에서는 직접 **파인튜닝**해 품질을 고정시킵니다!